# GRPO & verifiable rewards: hands-on (optional)
**LLMs, Animated · Lesson M3.5**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

You'll build every piece of a GRPO step in NumPy: a strict verifier, group-normalised advantages, and a policy update with a KL leash. The "policy" is a toy: a softmax over six candidate answers to one prompt, so each training step takes microseconds. Then you'll watch it learn, and watch it learn to **cheat** a sloppy verifier.

Stuck? Solutions are at the very bottom. Only `numpy` and `matplotlib` are needed.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

## 1 · A strict verifier
The prompt is *"What is 17 × 24? End with `Answer: <number>`."* The correct answer is 408.

Write `verify(text)` that returns **1.0** only if the text contains **exactly one** `Answer: <integer>` and that integer is 408, and **0.0** otherwise. Being strict matters: a lax checker is an invitation to reward hacking.

*(Hint: `re.findall(r"Answer:\s*(-?\d+)", text)` returns every number that follows `Answer:`.)*

In [ ]:
TARGET = 408

def verify(text):
    # TODO: 1.0 if exactly one "Answer: <int>" and it equals TARGET, else 0.0
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
assert verify("17 × 24 = 408. Answer: 408") == 1.0
assert verify("Answer: 418") == 0.0
assert verify("408") == 0.0, "no 'Answer:' tag means no reward"
assert verify("Answer: 408 Answer: 418") == 0.0, "two answers is a hedge, not an answer"
assert verify("Answer: 4080") == 0.0
print("✓ exercise 1")

## 2 · Group-relative advantages
For a group of rewards $r_1, \dots, r_G$ for the **same prompt**, GRPO uses

$$\hat A_i = \frac{r_i - \operatorname{mean}(r)}{\operatorname{std}(r) + \epsilon}$$

Use the population std (`np.std`, which divides by $G$). With `use_std=False`, return just $r_i - \operatorname{mean}(r)$ (the "Dr. GRPO" variant).

In [ ]:
def group_advantages(rewards, eps=1e-6, use_std=True):
    r = np.asarray(rewards, dtype=float)
    # TODO: centre by the group mean; if use_std, divide by (std + eps)
    raise NotImplementedError("Exercise 2")

**Check 2.** These are the lesson's worked and faded examples.

In [ ]:
a = group_advantages([1, 0, 0, 0])
assert np.allclose(a, [1.732, -0.577, -0.577, -0.577], atol=1e-3), a
assert np.allclose(group_advantages([1, 1, 0, 1, 0]), [0.816, 0.816, -1.225, 0.816, -1.225], atol=1e-3)
assert np.allclose(group_advantages([1, 1, 1, 1]), 0), "equal rewards carry no signal"
assert np.allclose(group_advantages([0, 0, 0]), 0)
assert abs(group_advantages([0.3, 0.9, 0.1, 0.5]).sum()) < 1e-9, "advantages always sum to zero"
assert np.allclose(group_advantages([1, 0, 0, 0], use_std=False), [0.75, -0.25, -0.25, -0.25])
print("✓ exercise 2")

## 3 · A toy policy and its group
Our "model" can only produce one of six complete responses. Its policy is a softmax over six **logits**. The reference model $\pi_\text{ref}$ is the starting policy, which prefers a wrong answer.

Implement `softmax(z)` (subtract the max for numerical stability) and `sample_group(logits, G, rng)`, which returns `G` sampled response indices.

In [ ]:
CANDS = ["Answer: 408",            # correct
         "Answer: 418",
         "Answer: 398",
         "Answer: 4080",
         "408",                    # right number, wrong format
         "Answer: 408 Answer: 418"]  # hedging: two answers
REF_LOGITS = np.array([0.0, 1.0, 0.5, 0.2, 0.8, 0.3])
print([verify(c) for c in CANDS])

In [ ]:
def softmax(z):
    # TODO
    raise NotImplementedError("Exercise 3a")

def sample_group(logits, G, rng):
    # TODO: draw G indices from softmax(logits), e.g. with rng.choice
    raise NotImplementedError("Exercise 3b")

**Check 3**

In [ ]:
p = softmax(REF_LOGITS)
assert abs(p.sum() - 1) < 1e-12 and np.all(p > 0)
assert np.allclose(softmax(np.array([1000.0, 1000.0])), [0.5, 0.5]), "subtract the max to avoid overflow"
idx = sample_group(REF_LOGITS, 20000, np.random.default_rng(1))
assert idx.shape == (20000,)
freq = np.bincount(idx, minlength=6) / len(idx)
assert np.allclose(freq, p, atol=0.01), (freq, p)
print("P(correct) at the start:", round(p[0], 3))
print("✓ exercise 3")

## 4 · One GRPO step
Each response here is a single "token", so the GRPO objective for one group is

$$\mathcal{J} = \frac{1}{G}\sum_{i=1}^{G} \hat A_i \log \pi(a_i) \;-\; \beta\, \mathrm{KL}(\pi \,\|\, \pi_\text{ref})$$

(With one gradient step per fresh batch, the PPO ratio is exactly 1, so clipping never activates and we can drop it.) For a softmax policy with logits $z$:

- $\dfrac{\partial \log \pi(a)}{\partial z} = \mathbf{e}_a - \pi$ (one-hot of $a$ minus the probabilities)
- $\dfrac{\partial\, \mathrm{KL}(\pi\|\pi_\text{ref})}{\partial z_k} = \pi_k\big(\log \pi_k - \log \pi_{\text{ref},k} - \mathrm{KL}\big)$

Implement `grpo_step`: sample a group, score it with `reward_fn`, compute advantages, and take one **gradient-ascent** step on $\mathcal{J}$. Return the new logits and the group's mean reward.

In [ ]:
def kl(p, q):
    return float(np.sum(p * (np.log(p) - np.log(q))))

def grpo_step(logits, ref_logits, rng, reward_fn=verify, G=8, lr=0.2, beta=0.04, use_std=True):
    pi, ref = softmax(logits), softmax(ref_logits)
    idx = sample_group(logits, G, rng)
    rewards = np.array([reward_fn(CANDS[i]) for i in idx])
    # TODO 1: advantages for the group
    # TODO 2: policy-gradient term: mean over the group of A_i * (onehot(a_i) - pi)
    # TODO 3: KL gradient: pi * (log pi - log ref - KL)
    # TODO 4: new_logits = logits + lr * (pg - beta * kl_grad)
    raise NotImplementedError("Exercise 4")
    return new_logits, rewards.mean()

**Check 4.** First a numerical check of the KL gradient (a habit worth keeping), then 300 training steps.

In [ ]:
z = np.array([0.3, -0.2, 0.9, 0.0, 0.1, -0.5]); ref = softmax(REF_LOGITS)
num = np.array([(kl(softmax(z + 1e-6 * np.eye(6)[k]), ref) - kl(softmax(z - 1e-6 * np.eye(6)[k]), ref)) / 2e-6 for k in range(6)])
pz = softmax(z)
assert np.allclose(pz * (np.log(pz) - np.log(ref) - kl(pz, ref)), num, atol=1e-6)

def train(steps=300, seed=0, **kw):
    r = np.random.default_rng(seed)
    logits, hist, probs = REF_LOGITS.copy(), [], []
    for _ in range(steps):
        logits, m = grpo_step(logits, REF_LOGITS, r, **kw)
        hist.append(m); probs.append(softmax(logits))
    return logits, np.array(hist), np.array(probs)

logits, hist, probs = train()
print("P(correct) after training:", round(softmax(logits)[0], 3))
assert softmax(logits)[0] > 0.9, "the correct answer should dominate"
assert hist[:20].mean() < 0.4 and hist[-50:].mean() > 0.9, "mean reward should climb"
plt.figure(figsize=(7, 3.5))
for k, c in enumerate(CANDS):
    plt.plot(probs[:, k], label=c)
plt.xlabel("GRPO step"); plt.ylabel("probability"); plt.legend(fontsize=8); plt.title("Policy over the six responses")
plt.show()
print("✓ exercise 4")

## 5 · The KL leash
The KL term pulls the policy back toward $\pi_\text{ref}$. Train with `beta=0` and with a large `beta=2.0`, and store the final $\mathrm{KL}(\pi\|\pi_\text{ref})$ and $P(\text{correct})$ of each run in the four variables below.

In [ ]:
# TODO: two runs of train(...), then fill these in
kl_free, p_free = None, None      # beta = 0
kl_tight, p_tight = None, None    # beta = 2.0

**Check 5**

In [ ]:
print(f"beta=0:   KL={kl_free:.2f}  P(correct)={p_free:.2f}")
print(f"beta=2.0: KL={kl_tight:.2f}  P(correct)={p_tight:.2f}")
assert kl_tight < kl_free, "a stronger leash keeps the policy closer to the reference"
assert p_tight < p_free
print("✓ exercise 5")

## Experiment · reward hacking, live
Swap the strict verifier for a sloppy one: *reward any response that contains "408"*. Three wrong or malformed responses now earn full reward. Which one does the policy end up preferring? Then try:
- `use_std=False` (Dr. GRPO style) and compare the curves;
- `G=2` vs `G=32`: how noisy is learning with tiny groups?

In [ ]:
def lax_verify(text):
    return 1.0 if "408" in text else 0.0

hacked, _, hprobs = train(reward_fn=lax_verify)
for c, p in sorted(zip(CANDS, softmax(hacked)), key=lambda t: -t[1]):
    print(f"{p:.2f}  {c!r}   strict reward: {verify(c)}")

## Recap
- A **verifier** is your reward function, so it is also your model's specification: anything it doesn't check, the policy may exploit.
- GRPO's advantage is the reward **relative to the group**: no value model, and no signal from groups whose rewards are all equal.
- The policy gradient raises $\log\pi$ for above-average samples and lowers it for below-average ones; the **KL term** limits how far it drifts from the reference.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
TARGET = 408

def verify(text):
    found = re.findall(r"Answer:\s*(-?\d+)", text)
    return 1.0 if len(found) == 1 and int(found[0]) == TARGET else 0.0

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def group_advantages(rewards, eps=1e-6, use_std=True):
    r = np.asarray(rewards, dtype=float)
    a = r - r.mean()
    return a / (r.std() + eps) if use_std else a

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()

def sample_group(logits, G, rng):
    return rng.choice(len(logits), size=G, p=softmax(logits))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def kl(p, q):
    return float(np.sum(p * (np.log(p) - np.log(q))))

def grpo_step(logits, ref_logits, rng, reward_fn=verify, G=8, lr=0.2, beta=0.04, use_std=True):
    pi, ref = softmax(logits), softmax(ref_logits)
    idx = sample_group(logits, G, rng)
    rewards = np.array([reward_fn(CANDS[i]) for i in idx])
    adv = group_advantages(rewards, use_std=use_std)
    onehots = np.eye(len(logits))[idx]
    pg = (adv[:, None] * (onehots - pi)).mean(axis=0)
    kl_grad = pi * (np.log(pi) - np.log(ref) - kl(pi, ref))
    new_logits = logits + lr * (pg - beta * kl_grad)
    return new_logits, rewards.mean()

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
l0, _, _ = train(beta=0.0)
l2, _, _ = train(beta=2.0)
kl_free, p_free = kl(softmax(l0), softmax(REF_LOGITS)), softmax(l0)[0]
kl_tight, p_tight = kl(softmax(l2), softmax(REF_LOGITS)), softmax(l2)[0]